# Gold Layer: Daily Sensor Aggregates

**Purpose:** Create daily Metrics for trends analysis and histroical reporting

**Why this table?**
- Dashboard shows TODAY's status (snapshot)
- This table shoes HISTORY (trends over time)
- Helps answer: "Is vibration getting worse?" "Was temerature stable?"

**Input:** `dev.silver.sensor_readings_enriched`

**Output:** `dev.gold.daily_sensor_metrics`

**Example Output:**
| Equipment | Sensor type | Date | Min | Max | Avg | Readings |
| --------- | ----------- | ---- | --- | --- | --- | -------- |
| EQ-0001   | temerature  | 2024-01-15 | 58.2 | 71.5 | 65.1 | 145 |
| EQ-0001   | vibration   | 2024-01-15 | 0.01 | 0.06 | 0.035 | 145 |

**Business Use Cases:**
- Trend analysis: Trend analysis: Is equipment degrading?
- Reporting: Show trends to executives
- ML features: Train models on historical patterns
- Anomaly detection: Detect unusaual daily patterns

## Configuration and Setup

In [0]:
from pyspark.sql.functions import (
    when, col, to_date, avg, min as spark_min, max as spark_max,
    count, stddev, round as spark_round
)

# Configuration
INPUT_TABLE =  "dev.silver.sensor_readings_enriched"
OUTPUT_TABLE = "dev.gold.daily_sensor_metrics"

print("=" * 60)
print("CONFIGURATION")
print("=" * 60)
print(f"Input: {INPUT_TABLE}")
print(f"Output: {OUTPUT_TABLE}")
print(f"This table creates daily summries for trend analysis")

## Load Data and Extract Date

In [0]:
# Load enriched data
enriched_df = spark.read.table(INPUT_TABLE)

# Extract DATE from timestramp (ignore time of day)
# Why? We want daily aggregates, not hourly or per-minute
# So "2024-01-15 10:30:45" and "2024-01-15 14:20:15"
# both belong to "2024-01-15"

enriched_with_date = enriched_df.withColumn(
    "reading_date",
    to_date(col("timestamp"))
)

print(f"Loaded {enriched_df.count()} sensor readings")
print(f"Date range in data:")
enriched_with_date.select(
    col("reading_date")
).distinct().orderBy("reading_date").show(10)

## Group by Equipment, Sensor Type, and Date

Create one row per equipment-sensor-day combination

In [0]:
# Group by three dimensions: equipment + sensor type + date
# Why three dimensions?
# - Equipment: Different machines have different characteristics
# - Sensor type: Temerature behaves differently than vibration
# - Date: We want separate rows for each day

daily_aggregates = enriched_with_date.groupBy(
    "reading_date",
    "equipment_id",
    "equipment_name",
    "equipment_type",
    "factory_location",
    "sensor_type"
).agg(
    # Minimum value for the day
    # Why min? shows lowest reading that day
    spark_round(spark_min("value"), 4).alias("min_value"),

    # Maximum valuefor the day
    # Why max? Shows highest reading that day (detects spikes)
    spark_round(spark_max("value"),4).alias("max_value"),

    # Average value for the day
    # Why avg? Shows typical behaviour that day
    spark_round(avg("value"),4).alias("avg_value"),

    # Standard deviation (spread of values)
    # Why stddev? High = unstable, Low = stable
    spark_round(stddev("value"), 4).alias("stddev_value"),

    # How many readings that day?
    # Why count? Shows data quality (gaps = missing readings)
    count("*").alias("reading_count")
)

print(f"Created daily aggregates")
print(f"Total records: {daily_aggregates.count()}")
print(f"Sample (first 10 equipment-sensor-day combinations):")
daily_aggregates.show(10, truncate=False)

## Classify Daily Sensor Health

Flag days where readings were abnormal

In [0]:
# Classify each day as NORMAL, WARNING, or ALERT
# Why classify? Makes it easy to spot bad days
# Instead of reading "avg_value=0.087", managers see "ALERT"

#Defind thresholds per sensor type
# These are based on typical equipment operating ranges

health_classified = daily_aggregates.withColumn(
    "daily_health",

    # Temperature classification
    when(col("sensor_type") == "temerature", 
         when(col("avg_value") > 75, "ALERT")  # Too hot
         .when(col("avg_value") > 70, "WARNING")  # Getting warn
         .otherwise("NORMAL")  # Good
    )
    # Vibration classification
    .when(col("sensor_type") == "vibration",
        when(col("avg_value") > 0.08, "ALERT")   # Too much vibration
        .when(col("avg_value") > 0.06, "WARNING")  # Increasing
        .otherwise("NORMAL")
    )
    # Pressure classification
    .when(col("sensor_type") == "power_consumption",
          when(col("avg_value") > 200, "ALERT") #Overloaded
          .when(col("avg_value") > 150, "WARNING") # High draw
          .otherwise("NORMAL")
    )
    .otherwise("UNKNOWN")
)

print(f"Daily health classification added")
print(f"\nDays by health status:")
health_classified.groupBy("sensor_type", "daily_health").count().show()


print(f"\nSample with health status:")
health_classified.select(
    "reading_date",
    "equipment_id",
    "sensor_type",
    "avg_value",
    "daily_health"
).show(15, truncate=False)

## Calculate Daily Variance

Detect if readings were stable or unstable that day

In [0]:
# Add stability indicatory
# Why stability? Fluctuating values = problems
# Smooth values = normal operation

# If stddev is high, values jumped around a lot
# If stddev is low, values stayed consistent

stability_added = health_classified.withColumn(
    "stability",

    # If stddev is very low, reading), mark as INSUFFICIENT_DATA
    when(col("stddev_value").isNull(), "INSUFFICIENT_DATA")

    # If stddev is very low,  readings were stable
    .when(col("stddev_value") < 1, "STABLE") # Very consistent

    # If stddeve is moderate, some fluctuation
    .when(col("stddev_value") <5, "NORMAL") # Some variaction

    # If stddev is high, readings jumped around
    .otherwise("UNSTABLE")
)

print(f"Stability classification added")
print(f"\nReading stability distribution:")
stability_added.groupBy("stability").count().show()

print(f"\nSample with stability:")
stability_added.select(
    "reading_date",
    "equipment_id",
    "sensor_type",
    "stddev_value",
    "stability"
).show(10, truncate=False)

## Add Metadata

In [0]:
from pyspark.sql.functions import current_timestamp, lit

# Add when this aggregation was calculated
final_daily = stability_added.withColumn(
    "metric_timestamp",
    current_timestamp()
).withColumn(
    "metric_source",
    lit("silver_sensor_readings_enriched") # Track data source
)

print(f"Metadata added")
print(f"Total daily meric records: {final_daily.count()}")

## Reorder Columns for Reporting

In [0]:
# Reorder columns: important ones first for reporting

dashboard_daily = final_daily.select(
    # Time and identity 
    "reading_date",
    "equipment_id",
    "equipment_name",
    "equipment_type",
    "sensor_type",

    # Key metrics
    "avg_value",
    "min_value",
    "max_value",
    "stddev_value",

    # Quality indicators
    "reading_count",
    "daily_health",
    "stability",

    # Metadata
    "factory_location",
    "metric_timestamp"
)

print(f"Columns reordered for reporting")

## Write to Gold Table

In [0]:
# Write daily aggregates to Gold layer
# This is historical data for reporting and analysis

print(f"Writing {dashboard_daily.count()} daily metrics..")

dashboard_daily.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(OUTPUT_TABLE)

print(f"Daily metrics table written!")
print(f"Table: {OUTPUT_TABLE}")
print(f"Records: {dashboard_daily.count()}")

## Verify and Analyze

In [0]:
# Read back and verify
daily_verify = spark.read.table(OUTPUT_TABLE)

print("=" * 80)
print("DAILY SENSOR METRICS (GOLD LAYER)")
print("=" * 80)

print(f"\n Total records: {daily_verify.count()}")
print(f"Date range: Form {daily_verify.agg(spark_min('reading_date')).collect()[0][0]} to {daily_verify.agg(spark_max('reading_date')).collect()[0][0]}")

print(f"\nSample daily metrics")
daily_verify.orderBy("reading_date", "equipment_id", "sensor_type").show(20, truncate=False)

## Business Analytics

In [0]:
# Show insights from daily metrics

print("=" * 80)
print("DAILY METRICS INSIGHTS FOR BUSINESS")
print("=" * 80)

# Which days had the most ALERT readings
print("\n DAYS WITH ALERT CONDITIONS:")
daily_verify.filter(col("daily_health") == "ALERT") \
    .groupBy("reading_date", "sensor_type") \
    .count() \
    .orderBy("reading_date", ascending=False) \
    .show(10)

# Equipment with most unstable readings
print("\n EQUIPMENT WITH MOST UNSTABLE READINGS:")
daily_verify.filter(col("stability") == "UNSTABLE") \
    .groupBy("equipment_id", "sensor_type") \
    .count() \
    .orderBy("count", ascending=False) \
    .show(10)

# Average readings per equipment per day
print("\n DATA QUALITY (READINGS PER EQUIPMENT PER DAY):")
daily_verify.groupBy("equipment_id").agg(
    spark_round(avg("reading_count"), 0).alias("avg_readings_per_day")
).orderBy("avg_readings_per_day", ascending=False) \
    .show(10)

# Days with concerning health status
print("\n CONCERNING DAYS (WARNING OR ALERT STATUS):")
daily_verify.filter(
    (col("daily_health") == "ALERT" )| (col("daily_health") == "WARNING")
).select(
    "reading_date",
    "equipment_id", 
    "sensor_type",
    "avg_value",
    "daily_health"
).orderBy("reading_date", ascending=False) \
    .show(15, truncate=False)

print("\n" + "=" * 80)

## Final Summary

In [0]:
print("=" * 80)
print("GOLD LAYER: DAILY SENSOR METRICS - COMPLETE!")
print("=" * 80)

print(f"""
    WHAT WE BUILD
    Daily aggregated sensor metrics for trend analysis

    TABLE STRUCTURE:
    One row = One equipment + One sensor + One day

    KEY METRICS:
    - min_value: Lowest readigns that day
    - max_value: Highest reading that day
    - avg_value: Average readings that day
    - stddev_value: How much values varied
    - reading_count: How many readings collected

    :
    - daily_health: NORMAL, WARNING, or ALERT
    - stability: STABLE, NORMAL, UNSTABLE, or INSUFFICIENT_DATA

    USE CASES:
    1. Trend analysis: Is equipment getting worse?
    2. Reporting: Daily summaries for executives
    3. Anomaly detection: Identify unusual days
    4. ML features: Historical partterns for models
    5. Compliance: Document equipment behaviour

    EXAMPLE QUESTION IT ANSWERS:
    "Was EQ-0001's temperature stable yesterday?"
    Query: SELECT * FROM {OUTPUT_TABLE}
    WHERE equipment_id='EQ-0001' and sensor_type='temperature'
    AND reading_date='2024-01-15'

    Answer:
    - avg_temperature: 65°C (normal)
    - stability: STABLE (consistent readings)
    - status: NORMAL (no issues)
    """)

print("=" * 80)